# Data Runner Quickstart

This notebook shows how a forged data runner iterates a forged dataset and applies arbitrary processing logic to each item.

What you will do:

- forge a runner from YAML
- process items with a local Python function
- save results as JSON and CSV
- connect the runner to a structured model example


In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / 'agentic_forge').exists()), NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os

RUNNER_DIR = PROJECT_ROOT / 'examples' / 'data_runner'
MATERIALS_DIR = RUNNER_DIR / 'materials'
STRUCT_MATERIALS = MATERIALS_DIR / 'structured_model'

from agentic_forge import forge
from agentic_forge.data_runner.forged import ForgedDataRunner


## Step 1: Forge the runner

`materials/data_runner.yaml` wraps a dataset material and sets default runtime options such as batch size, persistence path, and save format. The first printout is mainly there to show that the dataset key is already injected into each runtime item.


In [ ]:
runner: ForgedDataRunner = forge(MATERIALS_DIR / 'data_runner.yaml')
print(type(runner).__name__)
print(runner.dataset.key_field)


## Step 2: Run a local processing function

The first pass uses a pure Python function and saves the outputs as JSON. This is the simplest way to understand the runner abstraction before involving models or external services.


In [ ]:
def log_and_summarize(*, item):
    key = item['key']
    return {
        'key': key,
        'title': item['title'],
        'topic': item['topic'],
        'preview': item['text'][:30],
    }

json_output_path = MATERIALS_DIR / 'example_log_and_summ_outputs.json'
json_output_path.unlink(missing_ok=True)
json_results = runner.run(
    process_item=log_and_summarize,
    save_every=1,
    save_path=str(json_output_path),
    save_format='json',
    skip_keys=['item_001'],
)

print(json_results)
print(json_output_path.read_text())


## Step 3: Save the same run as CSV

The runner can persist the processed records in a different format without changing the dataset material itself. This keeps the runtime behavior configurable while the input dataset stays declarative.


In [ ]:
csv_output_path = MATERIALS_DIR / 'example_log_and_summ_outputs.csv'
csv_output_path.unlink(missing_ok=True)
csv_results = runner.run(
    process_item=log_and_summarize,
    save_every=1,
    save_path=str(csv_output_path),
    save_format='csv',
    skip_keys=['item_001'],
)

print(csv_results)


## Step 4: Connect the runner to a structured model

The last section shows how `process_item` can call another forged object, in this case a structured model. The notebook uses a local fallback by default so the section stays runnable even when live credentials are not configured.


In [ ]:
structured_model = forge(STRUCT_MATERIALS / 'structured_model.yaml')
print(list(structured_model.list_chains()))


def classify_news(*, item):
    key = item['key']
    if os.environ.get('AGENTIC_FORGE_RUN_LIVE_LLM') != '1':
        return {
            'key': key,
            'title': item['title'],
            'news_type': item.get('topic', 'other'),
            'reasoning': ['stubbed classification'],
        }

    try:
        response = structured_model.invoke(
            'classify',
            {'title': item['title'], 'text': item['text']},
        )
    except Exception as exc:
        print(f'Falling back to stub for {key}:', exc)
        response = {
            'news_type': item.get('topic', 'other'),
            'reasoning': ['stubbed classification'],
        }

    output = dict(response)
    output['key'] = key
    output['title'] = item['title']
    return output

final_output_path = MATERIALS_DIR / 'example_outputs.json'
final_output_path.unlink(missing_ok=True)
results = runner.run(
    process_item=classify_news,
    save_every=1,
    save_path=str(final_output_path),
    save_format='json',
)

print(results)
